# Data modeling 8: physical design and performance

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-08-physical-design-and-performance.ipynb)

Begin with a slow report and its filter predicates. Decide whether to reduce scanned files, compact small files, or precompute a common aggregate. Check that the report returns the same rows after the change.

The logical model says what a table means. The physical design decides how it is stored: which column it is
partitioned by, how its rows are ordered, how big its files are, which aggregates are precomputed, and which
data types hold its keys and amounts. Those choices decide how much data each query reads, and in the cloud,
what it costs.

The shop's real tables are too small to show this, so the lesson generates a year of synthetic order lines (one
million rows) as Delta tables and measures how many files each query has to open. Delta records statistics for
every file in its transaction log, and `DataFrame.inputFiles()` lists the files a query will read after Delta
has used them, so the counts below are exactly what Spark would scan. Timings vary between runs and are left
out.

**Prerequisites:** [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb) and [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb). [PySpark 13](../pyspark/pyspark-13-performance-tuning.ipynb) covers Spark-side tuning.

*Last reviewed: November 2026, against Apache Spark 4.0 and Delta Lake 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session with Delta Lake. The second loads `src_products` (the product catalogue) as Delta tables: extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import os
import re
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["TZ"] = "UTC"  # timestamps collected into Python stay in UTC, like the session below
time.tzset()

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.sources.default", "delta")  # `create table` makes Delta tables
    .config("spark.sql.warehouse.dir", tempfile.mkdtemp(prefix="modeling-"))  # where the tables are stored
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("FATAL")  # quiet Spark's own logging; errors still raise in Python


def run(script):
    """Run several SQL statements that each end with a semicolon: spark.sql() takes one statement at a time."""
    for statement in re.split(r";[ \t]*(?:--[^\n]*)?(?:\n|$)", script):
        if statement.strip():
            spark.sql(statement)


print("Spark", spark.version)

In [ ]:
run("""
create table src_products (
    sku string, product_name string, category string, department string, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
""")
spark.sql("show tables").show()

The next cell generates one million order lines spread over 2025. `hash()` makes the values look random but be the
same on every run. It caps files at 100,000 rows, so the table is written as about ten files, the way a real table
of many gigabytes is written as thousands. It also defines `files_read`, which counts the files a query must open,
and `scan_filters`, which pulls the scan's partition and data filters out of the query plan.

In [ ]:
import contextlib
import io

spark.conf.set("spark.sql.files.maxRecordsPerFile", 100000)  # small files, so that file skipping is visible

spark.sql("""
create table sales_source as
select id                                                              as line_id,
       date_add(date '2025-01-01', cast(pmod(hash(id), 365) as int))   as order_date,
       cast(pmod(hash(id * 7), 50000) as int)                          as customer_id,
       'SKU-0' || cast(1 + pmod(hash(id * 13), 8) as int)              as sku,
       cast(1 + pmod(hash(id * 17), 5) as int)                         as quantity,
       cast(pmod(hash(id * 19), 5000) / 100.0 as decimal(10, 2))       as net_amount
from range(1000000)
""")


def files_read(table, where):
    """Files the query must open after partition pruning and Delta's data skipping, out of the table's total."""
    used = len(spark.sql(f"select * from {table} where {where}").inputFiles())
    return f"{used} of {len(spark.table(table).inputFiles())} files"


def scan_filters(query):
    """The partition and data filters Spark pushes into the table scan, from the physical plan."""
    plan = io.StringIO()
    with contextlib.redirect_stdout(plan):
        spark.sql(query).explain()
    found = re.findall(r"(PartitionFilters|DataFilters): (\[[^\]]*\])", plan.getvalue())
    return {name: re.sub(r"#\d+", "", value) for name, value in found}  # drop Spark's internal column ids


spark.sql("select count(*) as rows, min(order_date) as first_day, max(order_date) as last_day from sales_source").show()

## How analytical engines read data

Lakehouse tables and warehouses store data **by column**, compressed, in large blocks: Parquet files and their
row groups under Delta and Iceberg, micro-partitions in Snowflake, blocks in BigQuery. For every file Delta
records the row count and the minimum and maximum of the first 32 columns in its transaction log. A query then
reads:

- only the **columns** it names, so a wide table costs nothing extra for columns a query skips;
- only the **files** whose min/max range could contain matching rows. Delta calls this **data skipping**; it
  happens while the query plans, before any data is read.

Two physical design levers follow from that. **Partitioning** puts each value of a column in its own directory,
so whole directories are skipped without looking at statistics. **Clustering** (sorting) keeps similar values
in the same files, so their min/max ranges are narrow and data skipping works. Both help only queries that
filter on those columns, so choose them from the filters your queries actually use, usually a date.

**Where it matters**
- Fact tables of more than a few gigabytes; cost control wherever you pay for bytes or compute scanned.

**Watch out for**
- Choosing a partition or cluster column that queries rarely filter on: all the cost of maintaining it, none of the benefit.

## Partitioning

A partitioned table stores each value of the partition column, usually a month or day of the event date, in its
own directory. A filter on that column becomes a **partition filter**, and Spark lists only the matching
directories: this is **partition pruning**.

Partition only large tables, by a column almost every query filters on, with a moderate number of values.
Databricks recommends not partitioning tables under about a terabyte and aiming for partitions of at least a
gigabyte; too many partitions (by customer, or by hour on a small table) produce thousands of tiny files that
every query pays to list and open. For smaller tables, sorting or liquid clustering does the job better.

**Where it matters**
- Large time-ranged facts and event tables, and lifecycle management (dropping or archiving old partitions).

**Watch out for**
- Partitioning by a high-cardinality column: the small-files problem.
- Partition columns that queries do not filter on directly, such as a month column when queries filter on the date.

**Quick check.** If reports normally filter by month, month partitions can let the engine skip other months. Partitioning by customer ID can instead create many tiny directories. Confirm the scan benefit in the plan or profile.

### Check whether the scan skips other months

Write a copy partitioned by month. Compare the plan for March with a month filter and with only a date predicate. Look for partition filters in the scan.

In [ ]:
spark.sql("""
create table sales_by_month partitioned by (order_month) as
select *, date_format(order_date, 'yyyy-MM') as order_month from sales_source
""")
month_filter = "order_month = '2025-03'"
date_filter = "order_date between '2025-03-01' and '2025-03-31'"
for label, where in [("partition column", month_filter), ("order_date only", date_filter)]:
    print(f"filter on the {label}: {files_read('sales_by_month', where)}")
    print("   ", scan_filters(f"select sum(net_amount) from sales_by_month where {where}"))

**What to notice:** Filtering on `order_month` became a **partition filter**: Spark listed only March's directory and opened its 2
files out of 24. Filtering on `order_date` asks the same question, but Spark does not know that `order_date`
and the directory name are related, so the plan has no partition filter, only a data filter. It still opened just
2 files, because Delta's per-file statistics showed that every other file's dates fall outside March. That is
data skipping, and it works on any column whose values are grouped in files, partitioned or not.

## Clustering and data skipping

Data skipping is only as good as the order of the rows. Write rows in random order and every file holds dates
from the whole year: every file's min/max range overlaps any date filter, and nothing can be skipped. Sorted by
date, each file covers a narrow range, and a one-week filter touches one or two files.

Delta offers three ways to keep data clustered:

- **Sort on write**: order the data by the filter column when writing, as the next example does.
- **`optimize ... zorder by (col1, col2)`**: rewrites files so rows close in several columns land together; it
  must be rerun as new data arrives.
- **Liquid clustering** (`create table ... cluster by (col1, col2)`, then `optimize`): Delta tracks clustering
  incrementally and lets you change the clustering columns later. It is the recommended default for new tables
  on Databricks and replaces partitioning plus Z-order for most of them.

Warehouses have the same idea under other names: Snowflake clustering keys, BigQuery clustered tables,
Redshift sort keys, Iceberg sort orders.

**Where it matters**
- Large facts filtered by date ranges, customer or tenant; dimensions looked up by key.

**Watch out for**
- Clustering decays as new unsorted data arrives; reclustering (`optimize`) costs compute and needs a schedule.

### Compare file ranges before and after sorting

Read the date min and max for each file. Narrower ranges make it easier for an engine to skip files when the query filters on a date outside those ranges.

In [ ]:
spark.sql("create table sales_unsorted as select * from sales_source")
spark.sql("create table sales_sorted as select * from sales_source order by order_date")

for table in ["sales_unsorted", "sales_sorted"]:
    log = spark.sql(f"describe detail {table}").first()["location"] + "/_delta_log/*.json"
    print(f"{table}: per-file statistics from the Delta log (first 3 files by date)")
    (spark.read.json(log).where("add is not null")
        .selectExpr("get_json_object(add.stats, '$.minValues.order_date') as min_date",
                    "get_json_object(add.stats, '$.maxValues.order_date') as max_date",
                    "get_json_object(add.stats, '$.numRecords') as num_records")
        .orderBy("min_date").show(3))

week = "order_date between '2025-06-02' and '2025-06-08'"
for table in ["sales_unsorted", "sales_sorted"]:
    print(f"one week in June from {table}: {files_read(table, week)}")

**What to notice:** In the unsorted table every file spans the whole year (2025-01-01 to 2025-12-31), so a one-week filter cannot rule
any of them out: all 10 files are read. In the sorted table each file covers about five weeks, and the same filter
reads 1 file of 11. Same data, same query, a tenth of the reading, purely from the order the rows were written in.

## Small files and compaction

Every write to a Delta table adds at least one file. A stream or a frequent small batch therefore creates
thousands of tiny files, and each query pays to list, open and read the footer of every one. **Compaction**
rewrites many small files into a few large ones: `optimize table_name` in Delta (Databricks can also do it
automatically with optimized writes and auto compaction). Like every Delta write, `optimize` is a new table
version; the old files stay for time travel until `vacuum` removes them.

**Where it matters**
- Streaming tables, frequently appended facts, and tables fed by many small `merge` statements.

**Watch out for**
- Compacting a table while it is being written: Delta's optimistic concurrency retries or fails the conflicting writer.

### Compact small files and check the same data

Write several tiny batches and count the files. Compact them, then verify that the rows still match. Treat elapsed time on this tiny fixture as illustrative.

In [ ]:
spark.sql("create table events_small (event_id int, event_type string)")
for batch in range(20):
    spark.sql(f"insert into events_small values ({batch}, 'click')")
print("files after 20 small inserts:", spark.sql("describe detail events_small").first()["numFiles"])
spark.sql("optimize events_small").selectExpr("metrics.numFilesRemoved", "metrics.numFilesAdded").show()
print("files after optimize:", spark.sql("describe detail events_small").first()["numFiles"])

**What to notice:** Twenty inserts made twenty files for twenty rows. `optimize` removed all 20 and added 1, so a query now opens one
file instead of twenty. On a real table the ratio is the same and the files are megabytes, not bytes: compaction is
routine maintenance for any table written in small pieces.

## Aggregate tables and materialized views

Dashboards ask the same coarse questions repeatedly: sales by day and product, by month and region. An
**aggregate table** stores the fact pre-summarized at that coarser grain, so those queries read thousands of
rows instead of millions. A **materialized view** is the same idea managed by the engine (Databricks, Snowflake,
BigQuery, Redshift), which keeps it current and can rewrite matching queries to use it automatically.

Rules that keep aggregates correct:

- Aggregate only **additive** measures, or store the components (sum and count, never an average).
- Keep the aggregate's grain explicit and its dimension keys conformed with the base fact, so it can be joined
  to the same dimensions.
- Refresh it with the same incremental, idempotent pattern as the fact (lesson 7): recompute affected days.
- Reconcile it against the base fact in tests (lesson 9).

**Where it matters**
- High-traffic dashboards, BI extracts, and any metric queried far more often than its data changes.

**Watch out for**
- Distinct counts in aggregates: they cannot be summed across rows; store a sketch (HyperLogLog) or recompute from the base.

### Check an aggregate against the base fact

Build one row per product and day. Roll it up to month, then compare the result with the same monthly query on the base fact. Keep enough detail to answer the filters users need.

In [ ]:
spark.sql("""
create table agg_sales_daily as
select order_date, sku, count(*) as lines, sum(quantity) as units, sum(net_amount) as net_sales
from sales_source group by all
""")
print(f"base table: {spark.table('sales_source').count():,} rows; daily aggregate: {spark.table('agg_sales_daily').count():,} rows")

monthly = """select cast(trunc(order_date, 'MM') as date) as month, {cols}
             from {table} where order_date < '2025-04-01' group by all order by month"""
spark.sql(monthly.format(table="sales_source", cols="count(*) as lines, sum(net_amount) as net_sales")).show()
spark.sql(monthly.format(table="agg_sales_daily", cols="sum(lines) as lines, sum(net_sales) as net_sales")).show()

**What to notice:** The aggregate has 2,920 rows (365 days × 8 SKUs) instead of 1,000,000, and gives identical monthly results,
because counts and sums add up across days. Note `sum(lines)`, not `count(*)`, when reading the aggregate: each of
its rows already stands for many lines.

## Keys and data types

Physical types matter more at scale than they look:

- **Join keys:** integer surrogate keys are smaller and faster to join and compare than long strings or
  composite natural keys. Hash keys (Data Vault) trade some of that for parallel loading.
- **Money:** use `decimal`, never floating point. Each floating-point addition rounds, so sums carry small
  errors, and because a distributed engine adds partial sums in whatever order tasks finish, the same query can
  return different last digits on different runs.
- **Dates and timestamps:** store event time as a timestamp in a known zone (usually UTC, as this session is
  configured) and derive the business date in the model, not in every report.
- **Low-cardinality text** (status, country) compresses well in columnar formats; a lookup table is rarely
  needed for storage reasons.

**Where it matters**
- Every fact table's DDL; reviewing models generated by tools that default to strings or doubles.

**Watch out for**
- Keys stored as different types in different tables (`int` here, `string` there): joins need casts and lose statistics.

### Use decimal for exact money arithmetic

Sum ten values of 0.1 using floating-point and decimal types. Expect exactly 1.0 from decimal. Choose a scale that fits the amounts and the rounding rule.

In [ ]:
spark.sql("""
select aggregate(array_repeat(cast(0.1 as double), 10), cast(0 as double), (total, x) -> total + x)   as double_sum,
       aggregate(array_repeat(cast(0.1 as decimal(10, 2)), 10), cast(0 as decimal(10, 2)),
                 (total, x) -> cast(total + x as decimal(10, 2)))                                   as decimal_sum
""").show(truncate=False)

**What to notice:** Ten additions of 0.1 as `double` give 0.9999999999999999, because 0.1 has no exact binary representation and
every addition rounds. The `decimal` sum is exactly 1.00. Over millions of rows the error grows, equality checks
between two totals fail, and in Spark the order of additions depends on task scheduling, so a `double` total can
change between runs with no change in the data.

## Physical design in each platform

| Platform | Skip partitions | Organize within | Notes |
|---|---|---|---|
| Delta Lake (this lesson) | partition columns (large tables only) | sort on write, Z-order, liquid clustering | data skipping from per-file statistics; `optimize` compacts and clusters |
| Iceberg | hidden partitioning by transforms (`days(ts)`) | sort order | queries filter on the column, not a derived partition column |
| Snowflake | (automatic micro-partitions) | clustering key | define a clustering key only for large tables filtered on a column the load order does not follow |
| BigQuery | `partition by` a date or integer range | `cluster by` up to 4 columns | billing by bytes scanned makes pruning a cost question; `require_partition_filter` enforces it |
| Redshift | (none) | sort key; distribution key | `distkey` co-locates join keys across nodes; `diststyle all` copies small dimensions to every node |

See the [Delta Lake](../delta/delta.ipynb), [Iceberg](../iceberg/iceberg.ipynb), [Snowflake](../snowflake/snowflake.ipynb),
[BigQuery](../bigquery/bigquery.ipynb) and [Redshift](../redshift/redshift.ipynb) notebooks for each platform's details.

## Stars, wide tables and the engine

Physical design also informs the logical shape. A star needs a join per dimension at query time; a wide table
(lesson 6) does not. Spark joins a large fact to small dimensions cheaply by **broadcasting** each dimension to
every executor, so the fact is never shuffled; Databricks, Snowflake and BigQuery do the same. For most
workloads a star therefore performs well and keeps the model flexible. Wide tables win when dimensions are too
large to broadcast, or when one known query must be as fast as possible, and they cost more to keep current when
dimension attributes change.

A practical rule: model as a star, measure, and add wide tables or aggregates as marts where a measured query
needs them. Physical design follows the queries, so it is decided after the model, not instead of it.

**Where it matters**
- Deciding between star schemas, wide tables and aggregates for a dashboard or a data product.

**Watch out for**
- Denormalizing everything up front for speed before measuring: you pay in flexibility for a gain you may not need.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A Delta table is partitioned by `order_month`, and a dashboard filters on `order_date`. Does pruning happen?

**A.** Not partition pruning: the plan has no partition filter because Spark does not relate `order_date` to `order_month`. Delta's data skipping can still skip files by their `order_date` statistics if the dates are grouped in files. Filter on the partition column too, or cluster by `order_date` instead.

**Q2.** Why does sorting a table by date make a date-range query cheaper, even without partitioning?

**A.** Each file's min and max dates become narrow, so Delta's data skipping can rule out every file whose range does not overlap the filter. Unsorted, every file spans all dates.

**Q3.** A team partitions a 2 GB table by `customer_id` (200,000 customers). What problem do they create?

**A.** Hundreds of thousands of tiny files: listing and opening them dominates query time, and compression and statistics suffer. Do not partition a table this small; cluster by customer instead.

**Q4.** A streaming job appends to a Delta table every minute. What maintenance does the table need?

**A.** Regular compaction with `optimize` (or auto compaction), and `vacuum` to delete the files that compaction replaced once they are older than the time-travel retention.

**Q5.** Why can two runs of the same Spark query over a `double` amount column return different totals?

**A.** Floating-point addition rounds at each step, so the result depends on the order of additions, and Spark combines partial sums in whatever order tasks finish. `decimal` sums are exact.

## Key takeaways

- Columnar tables are read by column and by file; physical design decides how many files a query opens.
- Partition only large tables, by a column almost every query filters on; avoid many tiny partitions.
- Delta skips files by their min/max statistics, so clustering (sorting, Z-order, liquid clustering) can cut reading tenfold.
- Compact small files with `optimize`; every write adds files.
- Aggregate tables store additive measures at a coarser, conformed grain; read them with `sum`, not `count`.
- Use integer surrogate keys and `decimal` for money; floating-point sums depend on the order of addition.

**Next:** Continue with [Data modeling 9: testing, metrics and change](data-modeling-09-testing-metrics-and-change.ipynb).